### 1.获取promoter和tf-seq,等信息

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('/home/zlab/boltzscan')
from Bio import SeqIO
from fimocistarget.fimo_cistarg import PyMEMESuiteCisTargetBuilder
from fimocistarget.cistarg_impl import filter_fimo_by_seq_and_overlap
import pandas as pd
from pathlib import Path
from utils.io_utils import get_boltz_res, pwm_to_meme, read_motif
from utils.utils import calc_ipsae
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import json

In [ ]:
extend_range = 5
task_name = 'rose'
fimo_pvalue_thresh = 1e-5
overlap_thresh = 0.5

pwm_dir = Path('/home/zlab/ms/odata/OB_rose_pwms')
tf2pwm_file = pwm_dir / 'tf2pwms.json'

with open(tf2pwm_file, 'r') as f:
    tf2pwm = json.load(f)

In [ ]:
pwm2tf = {}
for tf, pwms in tf2pwm.items():
    for pwm in pwms:
        if pwm not in pwm2tf:
            pwm2tf[pwm] = []
        pwm2tf[pwm].append(tf)

In [ ]:
tf_msa_dir = Path('data/MSA_DATABASE/rose_msa')
promoter_file = 'input_other/rose/Rosa_chinensis_promoter.fasta'
tf_msa_dct = {f.parts[-2]: f for f in tf_msa_dir.rglob('*0.a3m')}
pep_seqs = SeqIO.to_dict(SeqIO.parse('data/MSA_DATABASE/rose_msa/rose_pep.fasta', 'fasta'))
promoter_seqs = SeqIO.to_dict(SeqIO.parse(promoter_file, 'fasta'))

root_save_dir = Path('output')
task_specific_dir = root_save_dir / task_name
task_specific_dir.mkdir(exist_ok=True, parents=True)

### 在没有启动子序列的情况下，可以自己去提取

In [ ]:
# gene_id = 'RchiOBHm_Chr3g0471271'
# promoter_seqs[gene_id]

# with open(f'{gene_id}_promoter.fasta','w') as f:
#     SeqIO.write(promoter_seqs[gene_id], f, 'fasta')


### 2.nest the motifs and write to meme file

In [ ]:
# 这个是rose启动子的碱基频率
A = 3.082e-01
C = 1.918e-01
G = 1.918e-01
T = 3.082e-01

save_meme_dir = task_specific_dir / 'meme_pwms'
for t,ps in tf2pwm.items():
    for p in ps:
        try:
            pwm_f = pwm_dir / f'{p}.txt'
            m_info = read_motif(pwm_f)
            mid = m_info['name']
            pwm_matrix = m_info['motif']
            save_dir = save_meme_dir
            save_dir.mkdir(exist_ok=True, parents=True)
            pwm_to_meme(pwm=pwm_matrix, motif_id=p, nsites=100, output_file=save_dir / f'{p}.meme',
            background=[A,C,G,T])
        except:
             pass

### 3. for sequence scan with fimo
- 1. pvalue threshold 1e-54
- 2. motif_pseudo 0.1
- 3. custom_bg= {'A':A, 'C':C, 'G':G, 'T':T} # point to rose promoter background

In [ ]:
save_scan_dir = task_specific_dir / 'fimo_results'

fimo = PyMEMESuiteCisTargetBuilder(
    fasta_file=promoter_file,
    motif_dir = save_meme_dir,
    output_dir=save_scan_dir,
    pvalue_thresh=1e-4,
    max_stored_scores=500000,
    motif_pseudo=0.1,
    custom_bg= {'A':A, 'C':C, 'G':G, 'T':T} # point to rose promoter background
)
fimo.build_database()

In [ ]:
hit = pd.read_csv(f'{save_scan_dir}/fimo_results_raw.csv.gz')
print(f'Total hits: {hit.shape[0]}')
fimo_df = hit[hit['pvalue'] <= fimo_pvalue_thresh]
print(f'Hits after filtering with pvalue<{fimo_pvalue_thresh}: {fimo_df.shape[0]}')

In [ ]:
# 生成TF蛋白和启动子的序列字段
# fimo_df['tf_seq'] = fimo_df['motif_id'].apply(lambda x: str(tf_seqs[x].seq))
fimo_df['promoter_seq'] = fimo_df['sequence_name'].apply(lambda x: str(promoter_seqs[x].seq))
fimo_df['tf_name'] = fimo_df['motif_id'].map(pwm2tf)
fimo_df =  fimo_df.explode('tf_name')
fimo_df.shape

In [ ]:
fimo_df = filter_fimo_by_seq_and_overlap(fimo_df, overlap_thresh)
print(f'Hits after removing redundancy: {fimo_df.shape[0]}')

In [ ]:
fimo_df.to_csv(f'{save_scan_dir}/fimo_results_filtered_{fimo_pvalue_thresh}.csv.gz', index=False)

In [ ]:
fimo_df['boltz_name'] = fimo_df['tf_name'] + '-' + fimo_df['motif_id'] + '-' + fimo_df['sequence_name'] + '-' + fimo_df['start'].astype(str) + '-' + fimo_df['stop'].astype(str)
fimo_df['tf_name'].unique().shape

In [ ]:

fimo_df['tf_name_new'] = fimo_df['tf_name'].apply(lambda x: x.replace('.','_'))
fimo_df['msa_path'] = fimo_df['tf_name_new'].map(tf_msa_dct)

In [ ]:
fimo_df['tf_seq'] = fimo_df['tf_name'].map(pep_seqs)
fimo_df['tf_seq'] = fimo_df['tf_seq'].apply(lambda x: ''.join(x))
fimo_df.to_csv(f'{save_scan_dir}/fimo_results_filtered_with_seq_{fimo_pvalue_thresh}.csv.gz', index=False)

In [ ]:
from utils.io_utils import to_boltz_fasta, distribute_files
fimo_df = pd.read_csv('output/rose/fimo_results/fimo_results_filtered_with_seq_1e-05.csv.gz')
save_boltz_dir = task_specific_dir / 'boltzscan_input'
save_boltz_dir.mkdir(exist_ok=True, parents=True)
to_boltz_fasta(fimo_df, save_boltz_dir)

In [ ]:
distribute_files(save_boltz_dir.joinpath('BOLTZ_INPUT'), save_boltz_dir)

### 处理boltz predict后的结果，计算ipsae指标

In [ ]:
boltzscan_dir = Path('res/test_dma1/boltz_results_BOLTZ_INPUT/predictions')
res_dct = {}
for d in boltzscan_dir.iterdir():
    for f in d.glob("*.json"):
        with open(f, 'r') as f:
            data = json.load(f)
        res_dct[d.parts[-1]] = data
            
BOLTZSCAN_RES = pd.DataFrame(res_dct).T

In [ ]:
need_score = BOLTZSCAN_RES[['confidence_score','ptm', 'iptm']]

In [ ]:
def batch_calc_ipsae(res_dir):
    for d in tqdm(res_dir.iterdir(), desc='calc_ipsae'):
        fs = list(d.iterdir())
        for f in fs:
            if f.suffix == '.cif':
                cif = f
            elif 'pae' in f.name:
                pae = f
        calc_ipsae(cif_file=cif, pae_file=pae)

batch_calc_ipsae(boltzscan_dir)

In [ ]:
from utils.io_utils import get_boltz_res
from utils.utils import read_ipsae

ipsae_d = get_boltz_res(boltzscan_dir, res_type='ipsae')
ipsae_res = {}
for k,v in ipsae_d.items():
    ipsae_res[k] = read_ipsae(v)


In [ ]:
ipsae_value_dct = {}
pdcok_value_dct = {}
for i in ipsae_res.keys():
    d = ipsae_res[i]
    
    ipsae_value_dct[i] = d[(d['Type'] == 'max') & (d['Chn1']=='A')]['ipSAE'].min().item()
    pdcok_value_dct[i] = d[(d['Type'] == 'max') & (d['Chn1']=='A')]['pDockQ'].max().item()

In [ ]:
need_score['ipsae'] = need_score.index.map(ipsae_value_dct)
need_score['pDockQ'] = need_score.index.map(pdcok_value_dct)

need_score

In [ ]:
need_score['TF'] = need_score.index.str.split('-').str[0]
need_score['TG'] = need_score.index.str.split('-').str[-3]
for tg, df in need_score.groupby('TG'):
    df = df.loc[:,['ipsae','TF','TG']].sort_values(by='ipsae' ,ascending=False)
    df.to_csv(f'input_other/dma/boltzscan_score_{tg}.csv')